# Bechert 1997 Blade-Riblet Replication (Stage 5)

Calibration target: Bechert et al. 1997 JFM 338:59–87 Figure 5, blade riblet at h/s=0.5, t/s=0.02.
Hypothesis bound: peak DR within ±2 pp of 9.9% near s+ ≈ 17, crossover s+ ≈ 27 ± 3.

Per-sub-run inputs (rsynced by `scripts/pull_flat_plate_results.py`):
* `results/02-flat-plate-riblet-bechert/{riblet|smooth}-{s+}/postProcessing/wallShearStress/0/wallShearStress.dat`
* `results/02-flat-plate-riblet-bechert/{riblet|smooth}-{s+}/postProcessing/residuals/...` (convergence check)
* `.../system/{controlDict,fvSchemes}` (evidence-bundle citation)

Bechert reference: `data/bechert-1997-fig5/digitized.csv` (provisional anchor points — see that file's README before the final verdict).

In [ ]:
from __future__ import annotations
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO = Path('/opt/aero-research-platform')
RESULTS = REPO / 'results' / '02-flat-plate-riblet-bechert'
BECHERT_CSV = REPO / 'data' / 'bechert-1997-fig5' / 'digitized.csv'
S_PLUS_SWEEP = [5, 10, 15, 17, 20, 25, 30, 35, 40]
SURFACES = ['riblet', 'smooth']
# Bechert peak/crossover hypothesis bound (DO NOT move after seeing data).
PEAK_DR_TARGET = 9.9
PEAK_DR_TOLERANCE_PP = 2.0
PEAK_S_PLUS_TARGET = 17
PEAK_S_PLUS_TOLERANCE = 3
CROSSOVER_S_PLUS_TARGET = 27
CROSSOVER_S_PLUS_TOLERANCE = 3

In [ ]:
def load_wall_shear(case_dir: Path) -> pd.DataFrame:
    """Load wallShearStress.dat as iter, tau_x, tau_y, tau_z (patch-averaged).

    OpenFOAM v2412 wallShearStress FO emits (min, max, average) vectors per
    patch per time step. We keep the average only.
    """
    wss_root = case_dir / 'postProcessing' / 'wallShearStress'
    if not wss_root.exists():
        return pd.DataFrame()
    candidates = sorted(wss_root.glob('*/wallShearStress.dat'))
    if not candidates:
        return pd.DataFrame()
    rows = []
    text = candidates[-1].read_text().splitlines()
    for line in text:
        if not line.strip() or line.startswith('#'):
            continue
        # Numbers + parenthesized vectors. Pull all floats out.
        nums = [float(x) for x in re.findall(r'-?\d+\.?\d*(?:[eE][+-]?\d+)?', line)]
        if len(nums) >= 10:  # iter + 3*(min,max,avg)
            it = nums[0]
            tau_avg = nums[7:10]  # last vector triple = average
            rows.append((it, *tau_avg))
    return pd.DataFrame(rows, columns=['iter', 'tau_x', 'tau_y', 'tau_z'])


def converged_tail_drag(df: pd.DataFrame, tail_n: int = 2000) -> float:
    """Patch-averaged tau_x over the last `tail_n` iterations."""
    if df.empty:
        return float('nan')
    tail = df.tail(tail_n)
    return float(tail['tau_x'].mean())

In [ ]:
rows = []
for s_plus in S_PLUS_SWEEP:
    paired = {}
    for surface in SURFACES:
        case = RESULTS / f'{surface}-{s_plus}'
        df = load_wall_shear(case)
        paired[surface] = converged_tail_drag(df)
    tau_riblet, tau_smooth = paired['riblet'], paired['smooth']
    # DR% from patch-averaged tau_x: Cd = 2 * tau_x (non-dim, A_ref = A_patch)
    # so DR cancels A_ref and the factor of 2.
    if np.isfinite(tau_riblet) and np.isfinite(tau_smooth) and tau_smooth != 0.0:
        dr_pct = (tau_smooth - tau_riblet) / tau_smooth * 100.0
    else:
        dr_pct = float('nan')
    rows.append({'s_plus': s_plus, 'tau_riblet': tau_riblet, 'tau_smooth': tau_smooth, 'dr_percent': dr_pct})

measured = pd.DataFrame(rows)
measured

In [ ]:
bechert = pd.read_csv(BECHERT_CSV, comment='#')

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(bechert['s_plus'], bechert['dr_percent'], 'o-', color='black', label='Bechert 1997 Fig 5 (digitized)')
ax.errorbar(measured['s_plus'], measured['dr_percent'], yerr=PEAK_DR_TOLERANCE_PP, fmt='s', color='C0', label='Stage 5 (this work)', capsize=4)
ax.axhline(0, color='grey', lw=0.5)
ax.axhspan(PEAK_DR_TARGET - PEAK_DR_TOLERANCE_PP, PEAK_DR_TARGET + PEAK_DR_TOLERANCE_PP, color='lightyellow', alpha=0.5, label=f'±{PEAK_DR_TOLERANCE_PP} pp band on Bechert peak')
ax.set_xlabel('s+ = s u_tau / nu')
ax.set_ylabel('Drag reduction (%)')
ax.set_title('Bechert 1997 blade-riblet replication, h/s=0.5')
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()

In [ ]:
# PASS/FAIL verdict against the hypothesis bound.
if measured['dr_percent'].dropna().empty:
    verdict = 'NO_DATA'
    peak_meas = float('nan')
    s_plus_peak = float('nan')
    crossover_meas = float('nan')
else:
    peak_idx = measured['dr_percent'].idxmax()
    peak_meas = float(measured.loc[peak_idx, 'dr_percent'])
    s_plus_peak = float(measured.loc[peak_idx, 's_plus'])
    # Crossover: linear interp where DR crosses 0.
    falling = measured.sort_values('s_plus')
    crossover_meas = float('nan')
    for i in range(len(falling) - 1):
        a, b = falling.iloc[i], falling.iloc[i + 1]
        if np.isfinite(a['dr_percent']) and np.isfinite(b['dr_percent']):
            if a['dr_percent'] > 0 >= b['dr_percent']:
                frac = a['dr_percent'] / (a['dr_percent'] - b['dr_percent'])
                crossover_meas = float(a['s_plus'] + frac * (b['s_plus'] - a['s_plus']))
                break
    within_peak = abs(peak_meas - PEAK_DR_TARGET) <= PEAK_DR_TOLERANCE_PP and abs(s_plus_peak - PEAK_S_PLUS_TARGET) <= PEAK_S_PLUS_TOLERANCE
    within_crossover = np.isfinite(crossover_meas) and abs(crossover_meas - CROSSOVER_S_PLUS_TARGET) <= CROSSOVER_S_PLUS_TOLERANCE
    verdict = 'PASS' if within_peak and within_crossover else 'PARTIAL' if within_peak or within_crossover else 'FAIL'

print(f'Peak DR measured     : {peak_meas:.2f}% at s+ = {s_plus_peak:.1f}  (target {PEAK_DR_TARGET}% ± {PEAK_DR_TOLERANCE_PP} pp at s+ {PEAK_S_PLUS_TARGET} ± {PEAK_S_PLUS_TOLERANCE})')
print(f'Crossover s+ measured: {crossover_meas:.1f}  (target {CROSSOVER_S_PLUS_TARGET} ± {CROSSOVER_S_PLUS_TOLERANCE})')
print(f'Verdict              : {verdict}')

measured.to_csv(RESULTS / 'dr_vs_s_plus.csv', index=False)
print(f'Wrote: {RESULTS / "dr_vs_s_plus.csv"}')